In [1]:
import warnings

warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from collections import defaultdict
import joblib
from scipy.stats import entropy

In [3]:
train_df = pd.read_csv("train.csv", parse_dates=["create_time", "model_create_time"], index_col='order_id')
webstat_df = pd.read_csv("t1_webstat.csv", parse_dates=["date_time"]).sort_values('date_time')
y = train_df["is_callcenter"]

In [4]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 104595 entries, 1269921 to 1197777
Data columns (total 18 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   create_time         104595 non-null  datetime64[ns]
 1   good_id             104595 non-null  int64         
 2   price               104595 non-null  int64         
 3   utm_medium          104595 non-null  int64         
 4   utm_source          94145 non-null   float64       
 5   sessionkey_id       104595 non-null  int64         
 6   category_id         104595 non-null  int64         
 7   parent_id           104595 non-null  int64         
 8   root_id             104595 non-null  int64         
 9   model_id            104595 non-null  int64         
 10  is_moderated        104595 non-null  int64         
 11  rating_value        33741 non-null   float64       
 12  rating_count        51613 non-null   float64       
 13  description_length  104595 

In [5]:
web_grouped = webstat_df.groupby('sessionkey_id')

In [6]:
session_lengths = web_grouped['pageview_duration_sec'].sum()
session_lengths

sessionkey_id
109996122     90.0
110019268    165.0
110020180      0.0
110040418    545.0
110044482      0.0
             ...  
134626779     22.0
134627402    244.0
134628420     93.0
134628743    152.0
134629277     64.0
Name: pageview_duration_sec, Length: 328430, dtype: float64

In [7]:
webstat_df.loc[:, 'product_in_sale'] = webstat_df['product_in_sale'].fillna(0)

# Информация о сессиях

In [8]:
session_info = pd.DataFrame({'duration': session_lengths})

In [9]:
session_info['num_pages'] = web_grouped.count()['pageview_number']

In [10]:
session_info['page_time_avg'] = web_grouped.mean()['pageview_duration_sec'].fillna(0)

In [11]:
session_info['page_time_std'] = web_grouped.std()['pageview_duration_sec'].fillna(0)

In [12]:
session_info['num_goods'] = web_grouped.count()['good_id'].fillna(0)

In [13]:
session_info['avg_price'] = web_grouped.mean()['price'].fillna(0)

In [14]:
session_info['std_price'] = web_grouped.std()['price'].fillna(0)

In [15]:
session_info

,duration,num_pages,page_time_avg,page_time_std,num_goods,avg_price,std_price
sessionkey_id,,,,,,,
109996122,90.0,7,15.000000,6.723095,0,0.0,0.00000
110019268,165.0,3,82.500000,55.861436,1,2986.0,0.00000
110020180,0.0,1,0.000000,0.000000,1,4490.0,0.00000
110040418,545.0,11,54.500000,80.525013,4,578.0,170.89568
110044482,0.0,1,0.000000,0.000000,1,4490.0,0.00000
...,...,...,...,...,...,...,...
134626779,22.0,1,22.000000,0.000000,0,0.0,0.00000
134627402,244.0,7,40.666667,42.828340,1,411.0,0.00000
134628420,93.0,5,23.250000,19.448650,0,0.0,0.00000


# Информация о товарах

In [16]:
goods_info = pd.DataFrame()

In [17]:
goods_info.index = webstat_df['good_id'].dropna().astype(int).unique()

In [18]:
goods_info['avg_price'] = webstat_df.groupby('good_id').mean()['price'].fillna(0)

In [19]:
goods_info = goods_info.merge(train_df.groupby('good_id').agg('mean')[['rating_value', 'rating_count']].fillna(0), left_index=True, right_index=True, how='left').fillna(0)

In [20]:
goods_info

,avg_price,rating_value,rating_count
22312252,2523.900000,0.0,0.0
55614318,4606.625000,0.0,0.0
10547740,691.879433,0.0,0.0
28114543,430.000000,0.0,1.0
62752433,5089.000000,0.0,0.0
...,...,...,...
68152262,936.000000,0.0,0.0
32815111,402.000000,0.0,0.0
75233530,2032.000000,0.0,0.0
75233841,1736.000000,0.0,0.0


# Страницы

In [21]:
def multiclass_label_data(column_name):
    webstat_df[column_name] = webstat_df[column_name].astype('category')

    page_info = (
        pd.get_dummies(webstat_df[column_name],
                    prefix=f'{column_name}',
                    prefix_sep='_',
                    dtype='uint8')                               
        .groupby(webstat_df['sessionkey_id'], sort=False).max() 
    )
    
    return page_info.fillna(0).astype('category')

# Время

In [22]:
from sklearn.preprocessing import KBinsDiscretizer

In [23]:
from sklearn.compose import ColumnTransformer

In [24]:
n_time_bins = 6
disc = KBinsDiscretizer(n_bins=n_time_bins)

In [25]:
ct = ColumnTransformer([('time_encoding', disc, ['sec_before_order', 'sec_after_order'])])

In [26]:
def get_time_features(df, val=False):
    order_time = df[['create_time', 'sessionkey_id']]
    session_start_time = web_grouped.min()['date_time'].to_frame()
    session_start_time.columns = ['start_time']
    session_end_time = web_grouped.max()['date_time'].to_frame()
    session_end_time.columns = ['end_time']
    time_features = order_time.merge(session_start_time, left_on='sessionkey_id', right_index=True, how='left').merge(
        session_end_time, left_on='sessionkey_id', right_index=True, how='left').drop(columns='sessionkey_id')
    time_features['sec_before_order'] = (time_features['create_time'] - time_features['start_time']).dt.seconds
    time_features['sec_after_order'] = (time_features['end_time'] - time_features['create_time']).dt.seconds
    time_features.drop(columns=['start_time', 'end_time', 'create_time'], inplace=True)
    
    if val:
        tf = ct.transform(time_features.fillna(0))
    else:
        tf = ct.fit_transform(time_features.fillna(0))
        
    enc_tf = pd.DataFrame.sparse.from_spmatrix(tf)
    enc_tf.index = time_features.index
    enc_tf.columns = [f'sec_before_order_{i}' for i in range(1, 1 + n_time_bins)] + [f'sec_after_order_{i}' for i in range(1, 1 + n_time_bins)]
    
    return enc_tf.sparse.to_dense().astype('category')

In [27]:
tf = get_time_features(train_df)
tf

,sec_before_order_1,sec_before_order_2,sec_before_order_3,sec_before_order_4,sec_before_order_5,sec_before_order_6,sec_after_order_1,sec_after_order_2,sec_after_order_3,sec_after_order_4,sec_after_order_5,sec_after_order_6
order_id,,,,,,,,,,,,
1269921,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1270034,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1268272,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1270544,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1270970,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1250981,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1173775,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1180920,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


# Итоговый код

In [28]:
def enrich_dataset(df, val=False):
    page_type_info = multiclass_label_data('page_type')
    X = df.merge(page_type_info, left_on='sessionkey_id', right_index=True, how='left')
    time_features = get_time_features(X, val)
    X = X.merge(time_features, left_index=True, right_index=True, how='left')
    
    y = X['is_callcenter'] if 'is_callcenter' in X.columns else None
    X = X[['sessionkey_id', 'good_id'] + list(page_type_info.columns) + list(time_features.columns)]
    X = X.merge(session_info, left_on='sessionkey_id', right_index=True, how='left').merge(goods_info, left_on='good_id', right_index=True, how='left')
    
    X.fillna(0, inplace=True)
    X.drop(columns=['sessionkey_id', 'good_id'], inplace=True)
    return X, y

In [29]:
X, y = enrich_dataset(train_df)

In [30]:
from sklearn.preprocessing import PolynomialFeatures

In [31]:
X.shape

(104595, 35)

In [32]:
test_df = pd.read_csv('test.csv', parse_dates=["create_time", "model_create_time"], index_col='order_id')

In [33]:
X_test, _ = enrich_dataset(test_df, True)

In [34]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 104595 entries, 1269921 to 1197777
Data columns (total 35 columns):
 #   Column              Non-Null Count   Dtype   
---  ------              --------------   -----   
 0   page_type_1         104595 non-null  category
 1   page_type_2         104595 non-null  category
 2   page_type_3         104595 non-null  category
 3   page_type_4         104595 non-null  category
 4   page_type_5         104595 non-null  category
 5   page_type_6         104595 non-null  category
 6   page_type_7         104595 non-null  category
 7   page_type_8         104595 non-null  category
 8   page_type_9         104595 non-null  category
 9   page_type_10        104595 non-null  category
 10  page_type_11        104595 non-null  category
 11  page_type_12        104595 non-null  category
 12  page_type_13        104595 non-null  category
 13  sec_before_order_1  104595 non-null  category
 14  sec_before_order_2  104595 non-null  category
 15  sec_before_orde

In [35]:
from sklearn.linear_model import LogisticRegressionCV
base_model = LogisticRegressionCV(class_weight='balanced',
                           solver='lbfgs', max_iter=10000, n_jobs=-1)

In [36]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer


def build_model(base_model):
    pipe = Pipeline([
        ('scale_numeric', ColumnTransformer(
            [('scaler', StandardScaler(), list(X.select_dtypes(include=np.number).columns)),
             ('identity', FunctionTransformer(lambda X: X), list(X.select_dtypes(include='category').columns))]
        )),
        ('interact', PolynomialFeatures(interaction_only=True, include_bias=True)),
        ('clf', base_model)
    ])

    return pipe

In [37]:
model = build_model(base_model)

In [38]:
X.shape

(104595, 35)

In [39]:
model

,steps,"[('scale_numeric', ...), ('interact', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaler', ...), ('identity', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


def solve(X, y, model):
    model.set_params(warm_start=True)
    
    X_train, X_val, y_train, y_val = train_test_split(X, y)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    
    print(roc_auc_score(y_val, y_pred))
    model.fit(X_val, y_val)
    
    return model

def submit(X_test, model):
    test_preds = model.predict_proba(X_test)[:, 1]

    pd.DataFrame({
        "order_id": test_df.index,
        "is_callcenter": test_preds
    }).to_csv("submission.csv", index=False)

In [ ]:
solve(X, y, model)

0.9386294940565555


KeyboardInterrupt: 

In [ ]:
model['clf'].coef_.shape

(1, 631)

In [ ]:
for page_type in webstat_df['page_type'].unique():
    idx = X[f"page_type_{page_type}"] == 1
    subset = X[idx]
    m = np.mean(y[idx])
    print(f"page_type_{page_type} {m} {'' if  0.1 <= m <= 0.9 else '!!!!!!!!!!!!'}")

page_type_2 0.33264902792885265 
page_type_7 0.2878088716880024 
page_type_1 0.35446855537538347 
page_type_3 0.1355652026139752 
page_type_9 0.2700778941713672 
page_type_4 0.2490402058811121 
page_type_10 0.28153018529587565 
page_type_5 0.25975997686524005 
page_type_12 0.25240384615384615 
page_type_8 0.23968755015782997 
page_type_6 0.03190981748777287 !!!!!!!!!!!!
page_type_11 0.06818181818181818 !!!!!!!!!!!!
page_type_13 0.215625 


In [ ]:
for page_type in webstat_df['page_type'].unique():
    m = X[f"page_type_{page_type}"].astype(float).mean()
    print(f"page_type_{page_type} {m} {'' if  0.1 <= m <= 0.9 else '!!!!!!!!!!!!'}")

page_type_2 0.6240546871265357 
page_type_7 0.12845738324011663 
page_type_1 0.9256752234810459 !!!!!!!!!!!!
page_type_3 0.6188536736937712 
page_type_9 0.07118887136096372 !!!!!!!!!!!!
page_type_4 0.22661695109708876 
page_type_10 0.015995028443042212 !!!!!!!!!!!!
page_type_5 0.13224341507720255 
page_type_12 0.00795449113246331 !!!!!!!!!!!!
page_type_8 0.17869879057316315 
page_type_6 0.3205889382857689 
page_type_11 0.00042067020412065587 !!!!!!!!!!!!
page_type_13 0.0030594196663320428 !!!!!!!!!!!!
